## UCB for Multi-Armed Bandits

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from pprint import pprint as pp

In [4]:
styles = []
efforts = []#,"medium","high"]
styles = [style + '_' + effort for effort in efforts for style in styles]

results_paths: List[str] = [
    f'../src/optimal_explorer/strategies/bandits/logs/game_results/style{style}.jsonl'
    for style in styles
]
ucb_path: str = f'../src/optimal_explorer/strategies/bandits/logs/ucb_mab.jsonl'
random_path: str = f'../src/optimal_explorer/strategies/bandits/logs/Random.jsonl'


models = [
    # "Gemini Pro 2.5",
    # "DeepSeek R1",
    # "Claude Opus 4",
    # "Claude 3.5 Sonnet",
    # "OpenAI o3",
]

model_ids = [
    # "google/gemini-2.5-pro-preview",
    # "deepseek/deepseek-r1-0528",
    # "anthropic/claude-opus-4",
    # "anthropic/claude-3.5-sonnet",
    # "openai/o3",
]

In [5]:
# results = []

# for style, results_path in zip(styles, results_paths):
#     with open(results_path, 'r') as file:
#         for line in file:
#             data = json.loads(line)
#             regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
#             model = data['model']
#             results.append({
#                 'game_id': data['game_id'],
#                 'model': model + f' (s={style})',
#                 'regret': regret,
#                 'length': len(data['history']),
#                 'style': style
#             })
# results_df = pd.DataFrame(results)

ucb = []
random = []

with open(ucb_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        # regret = [sum(data['regret_per_attempt'][:i]) for i in range(data['num_attempts'])]
        regret = data['regret_per_attempt']
        ucb.append({
            'game_id': data['game_id'],
            'model': 'UCB',
            'regret': regret,
            'length': data['num_attempts'],
        })
ucb_df = pd.DataFrame(ucb)

with open(random_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        # regret = [sum(data['regret_per_attempt'][:i]) for i in range(data['num_attempts'])]
        regret = data['regret_per_attempt']
        random.append({
            'game_id': data['game_id'],
            'model': 'Random',
            'regret': regret,
            'length': data['num_attempts'],
        })
random_df = pd.DataFrame(random)

In [7]:
# results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
ucb_df = ucb_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
random_df = random_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [8]:
ucb_df.head(5)

,game_id,model,regret,length
0,0,UCB,"[0.16637586244509472, 0.0, 0.1124259903007756,...",50
1,1,UCB,"[0.3033024887395841, 0.0, 0.7202101186248132, ...",50
2,2,UCB,"[0.11366757573670538, 0.5237362460508178, 0.0,...",50
3,3,UCB,"[0.3421490517730792, 0.18479913172954987, 0.60...",50
4,4,UCB,"[0.005654520951207664, 0.425452110789162, 0.0,...",50


In [9]:
random_df.head(5)

,game_id,model,regret,length
0,0,Random,"[0.1124259903007756, 0.29153456703351477, 0.0,...",50
1,1,Random,"[0.573568602625045, 0.41799192081031833, 0.0, ...",50
2,2,Random,"[0.11434008526043227, 0.0, 0.12929467579122012...",50
3,3,Random,"[0.6020422154347104, 0.3821193491499917, 0.184...",50
4,4,Random,"[0.0, 0.2749555353676135, 0.2749555353676135, ...",50


In [10]:
cumulative_regrets = []
error_bars = []

# for style in styles:
#     for mi, model in enumerate(models):
#         model_df = results_df[results_df['model'] == model_ids[mi] + f' (s={style})']
#         # Convert regret lists to numpy array for easier computation
#         regret_array = np.array(model_df['regret'].values.tolist())
        
#         # Calculate mean regret per turn
#         model_regret = np.mean(regret_array, axis=0)
#         cumulative_regret = np.cumsum(model_regret)
#         cumulative_regrets.append(cumulative_regret)
        
#         # Calculate standard error of the mean for each turn
#         sem = np.std(regret_array, axis=0) / np.sqrt(len(model_df))
#         cumulative_sem = np.cumsum(sem)
#         error_bars.append(cumulative_sem)
        
#         print(f'{model} cumulative regret: {cumulative_regret}')

ucb_regret_array = np.array(ucb_df['regret'].values.tolist())
ucb_regret = np.mean(ucb_regret_array, axis=0)
ucb_cumulative_regret = np.cumsum(ucb_regret)
# Calculate standard error of the mean for each turn
sem = np.std(ucb_regret_array, axis=0) / np.sqrt(len(ucb_df))
ucb_cumulative_sem = np.cumsum(sem)

random_regret_array = np.array(random_df['regret'].values.tolist())
random_regret = np.mean(random_regret_array, axis=0)
random_cumulative_regret = np.cumsum(random_regret)
# Calculate standard error of the mean for each turn
sem = np.std(random_regret_array, axis=0) / np.sqrt(len(random_df))
random_cumulative_sem = np.cumsum(sem)

In [16]:
fig = go.Figure()

colors = [px.colors.qualitative.Dark24[i] for i in [1, 10, 6, 15, 19]]

latex_font = dict(
    family="Latin Modern Roman, Times New Roman, serif",
    size=14,
    color="black"
)

x_ucb = list(range(len(ucb_cumulative_regret)))
x_random = list(range(len(random_cumulative_regret)))

# UCB shaded error region
fig.add_trace(go.Scatter(
    x=x_ucb + x_ucb[::-1],
    y=(ucb_cumulative_regret + ucb_cumulative_sem).tolist() + (ucb_cumulative_regret - ucb_cumulative_sem)[::-1].tolist(),
    fill='toself',
    fillcolor='rgba(57,106,177,0.18)',  # blue, low alpha
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False,
    name='UCB error'
))

# UCB main line
fig.add_trace(go.Scatter(
    x=x_ucb,
    y=ucb_cumulative_regret,
    mode='lines+markers',
    name='UCB',
    line=dict(color='blue', width=4),
    marker=dict(size=6, color='blue')
))

# Random shaded error region
fig.add_trace(go.Scatter(
    x=x_random + x_random[::-1],
    y=(random_cumulative_regret + random_cumulative_sem).tolist() + (random_cumulative_regret - random_cumulative_sem)[::-1].tolist(),
    fill='toself',
    fillcolor='rgba(204,37,41,0.18)',  # red, low alpha
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False,
    name='Random error'
))

# Random main line
fig.add_trace(go.Scatter(
    x=x_random,
    y=random_cumulative_regret,
    mode='lines+markers',
    name='Random',
    line=dict(color='red', width=4),
    marker=dict(size=6, color='red')
))

fig.update_layout(
    title='',
    xaxis_title='Step (Gaussian MAB, 5 arms, medium noise)',
    yaxis_title='Cumulative Regret',
    font=latex_font,
    legend=dict(
        title='Strategies',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='black',
        borderwidth=1,
        x=0.01,
        y=0.99,
        xanchor='left',
        yanchor='top'
    ),
    template='plotly_white',
    width=700,
    height=500,
    yaxis=dict(
        title='Cumulative Regret',
        side='right',
        tickmode='linear'
    ),
    xaxis=dict(
        tickmode='linear',
        dtick=10
    )
)
fig.show()

In [15]:
# # Use Plotly's Dark24 color set for darker colors


# # Set LaTeX font for all text elements
# latex_font = dict(
#     family="Latin Modern Roman, Times New Roman, serif",
#     size=14,
#     color="black"
# )

# fig = go.Figure()

# # for si, style in enumerate(styles):
# #     for mi, model in enumerate(models):
# #         x_vals = list(range(1, 50))
# #         y_mean = cumulative_regrets[mi + si]
# #         y_err = error_bars[mi + si]
# #         color = colors[(mi + si) % len(colors)]

# #         # Add shaded error region (as a filled area)
# #         fig.add_trace(go.Scatter(
# #             x=x_vals + x_vals[::-1],
# #             y=(y_mean + y_err).tolist() + (y_mean - y_err)[::-1].tolist(),
# #             fill='toself',
# #             fillcolor=f'rgba{tuple(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + (0.18,)}',
# #             line=dict(color='rgba(255,255,255,0)'),
# #             hoverinfo="skip",
# #             showlegend=False,
# #             name=f"{model} (s={style})"
# #         ))

# #         # Add main mean curve, thicker
# #         fig.add_trace(go.Scatter(
# #             x=x_vals,
# #             y=y_mean,
# #             mode='lines+markers',
# #             name=model + f' (s={style})',
# #             line=dict(width=4, color=color),
# #             marker=dict(size=6, color=color)
# #         ))

# # Add Bayes Optimal shaded error region (as a filled area)
# fig.add_trace(go.Scatter(
#     x=list(range(1, 50)) + list(range(1, 50))[::-1],
#     y=(ucb_cumulative_regret + ucb_cumulative_sem).tolist() + (ucb_cumulative_regret - ucb_cumulative_sem)[::-1].tolist(),
#     fill='toself',
#     fillcolor=f'rgba(0, 0, 0, 0.5)',
#     line=dict(color='rgba(255,255,255,0)'),
#     hoverinfo="skip",
#     showlegend=False,
#     name='Bayes Optimal'
# ))
# # Add Bayes Optimal line
# fig.add_trace(go.Scatter(
#     x=list(range(1, 50)),
#     y=ucb_cumulative_regret,
#     mode='markers+lines+lines',
#     name='Bayes Optimal',
#     line=dict(width=4, color='rgba(0, 0, 0, 0.8)'),
#     marker=dict(size=6, color='rgba(0, 0, 0, 0.8)')
# ))

# # Add y=x baseline as a dashed line
# # baseline_x = list(range(1, 50))
# # baseline_y = list(range(1, 50))
# fig.add_trace(go.Scatter(
#     x=baseline_x,
#     y=baseline_y,
#     mode='lines',
#     name='Baseline',
#     line=dict(color='black', width=2, dash='dash'),
#     showlegend=True
# ))

# fig.update_layout(
#     width=600,
#     height=470,
#     title=dict(
#         text='',
#         font=latex_font
#     ),
#     xaxis_title="Episode (Symbol Combo-Lock)",
#     yaxis_title="Cumulative Regret",
#     font=latex_font,
#     xaxis=dict(
#         tickmode='linear',
#         dtick=1,
#         ticks='outside',
#         showline=True,
#         mirror=True,
#         title_font=latex_font,
#         tickfont=latex_font
#     ),
#     yaxis=dict(
#         # range=[1, 7],
#         tickmode='linear',
#         dtick=1,
#         ticks='outside',
#         showline=True,
#         mirror=True,
#         side='right',  # default, but we want ticks on both sides
#         showticksuffix='all',
#         showticklabels=True,
#         title_font=latex_font,
#         tickfont=latex_font
#     ),
#     yaxis2=dict(
#         overlaying='y',
#         side='right',
#         tickmode='linear',
#         dtick=1,
#         ticks='outside',
#         showline=True,
#         showticklabels=True,
#         title_font=latex_font,
#         tickfont=latex_font
#     ),
#     legend=dict(
#         title='',
#         x=0.03,  # left edge, inside plot
#         y=0.97,  # top edge, inside plot
#         xanchor='left',
#         yanchor='top',
#         bgcolor='rgba(255,255,255,0.85)',
#         bordercolor='black',
#         borderwidth=1,
#         font=latex_font
#     ),
#     template='plotly_white'
# )

# # Add yaxis2 to all traces so ticks show on both sides
# for trace in fig.data:
#     trace.update(yaxis='y')